#### Tutorial 1, Problem 1: HIV Infection — Population Balances, No Reactor (dt)  

  System of 3 coupled ODEs, independent variable is TIME (not volume or weight)  
   Dependent Variable Vector z: [H, I, V]  
    Rate constants kr = [kr1,...,kr6] passed via args=(kr,)  
     Function signature: hiv(time, z, kr), time comes FIRST  
      Balances: Positive terms = population increase, negative terms = decrease  
      dH/dt = kr1 - kr2 H - kr3 H V  
      dI/dt = kr3 H V - kr4 I  
      dV/dt = -kr3 H V - kr5 V + kr6 I  
       Initial conditions: [H0, I0, V0] = [1e6, 0, 100]  
       Time span starts at 0 (initial value problem)  
       Plot with log y-axis, populations span many orders of magnitude (ylim 1 to 1e8)  



#### Tutorial 1, Problem 2: Gas PFR — Isothermal, Irreversible, Single Reaction (dV)  

  Gas phase, A -> B + C, first order in A, isothermal, no pressure drop  
   Dependent Variable Vector z: [FA, FB, FC]  
    Volumetric flow: NOT constant (moles increase, delta n = +1), so use F_i/F_tot, not v0  
     Concentrations: C_i = C_tot_0 * (F_i/F_tot), with C_tot_0 = P_0/(R T_0)  
      Drop (P/P_0) and (T_0/T) since isobaric and isothermal  
       Kinetics: k(T) from Arrhenius using a reference temperature  
       k = k(Tr) exp(Ea/R (1/Tr - 1/T))  
       Rates: r_A = -k C_A,  r_B = r_C = -r_A  
        Mole balances: dFi/dV = r_i  
        Use R = 8.314 J/(mol K) with P in Pa and T in K  
         Initial flow: Work out F_A0 from the production target, F_B = F_A0 X  
         so F_A0 = F_B/X  (convert kg/year to mol/s using the molar mass)  
         Initial conditions: [F_A0, 0, 0]  
          Find target volume: argmin(abs(conversion - x_target)), then use the same index for everything else  
          Make sure max volume is large enough to reach the target conversion!  



#### Tutorial 1, Problem 3: Gas PBR — Isothermal, Pressure Drop (Ergun, alpha given) (dW)  

  Packed-bed catalyst weight (W), isothermal, single reaction A + 1/2 B -> C  
   Dependent Variable Vector z: [FA, FB, FC, FI, p_r]  
    Inert (N2) MUST be in z and in F_tot, but dFI/dW = 0  
     Stoichiometry: r_B = 0.5 r_A,  r_C = -r_A,  r_I = 0  
      Feed: Stoichiometric, F_B0 = F_A0/2,  F_I0 = F_B0 * (79/21) (air)  
      Use the flow PER TUBE (total flow / number of tubes), so W is per tube  
       Rate law is in partial pressures: -r_A' = k P_A^(1/3) P_B^(2/3)  
       Rewrite with concentrations: r_A = -k R T C_A^(1/3) C_B^(2/3)  
       The R cancels, but use the SAME R in C_tot_0 and in the rate  
        Concentrations: C_i = C_tot_0 * (F_i/F_tot) * p_r,  C_tot_0 = P_0/(R T_0)  
        Pressure drop: dp_r/dW = -alpha/(2 p_r) * (F_tot/F_tot_0)  
        Pass F_tot_0, T_0, P_0 via args  
         Initial conditions: [F_A0, F_B0, 0, F_I0, 1.0]  (p_r starts at 1)  
         Watch out: dp_r/dW has 1/p_r, so keep the weight span short enough that p_r never reaches 0  
         Units: k must match the pressure units used (atm here), and R must match too



#### Tutorial 2, Problem 1: Liquid PFR — Adiabatic, Reversible (dV)  

  Liquid phase, no pressure drop, reversible, adiabatic ($Q = 0$)  
   Dependent Variable Vector z: [Fi,....., T]  
    Volumetric flow: Constant (v = v0 = F_A0/C_A0), passed via args=(v0,)  
     Concentrations: C_i = F_i / v0  
      Kinetics: Both k(T) (Arrhenius) and K_C(T) (Van 't Ho) vary with T  
       Energy Balance: dTdV = (r_A \Delta H)/(\sum F_i C_pi) 

       
       
#### Tutorial 2, Problem 2: Gas PFR — Non-Isothermal, Parallel Reactions, Heat Transfer Fluid (dV)  

  Gas phase, parallel reactions (r_1, r_2),  
    External cooling/heating fluid (T_a)  
      Dependent Variable Vector z: [Fi,....., T]  
      Add Ta to z if coolant temperature isn't constant!  
        Volumetric flow: Changes with temperature and concentration!  
        v = v0 * (F_tot/F_tot_0)* (T/T0)* (P_0/P) (Drop P_0/P if isobaric)  
          Parallel Mole Balances: Combine rates for shared species  
          e.g., dFAdV = r_1A + r_2A  
            Energy Balance: Add heat transfer term + sum heats of all reactions:  
            dTdV = ((r_A \Delta H)+Ua(T_a - T))/(\sum F_i C_pi)  
            Parallel units, hence r_A \Delta H --> (r_A1 \Delta HA1 + r_A2 \Delta HA2)  
            MAKE SURE THAT THE RATE AND DELTA H ARE BOTH BASED ON THE SAME COMPONET, eg r_ca and Hc1 



#### Tutorial 2, Problem 3: (Isothermal Gas Pressure Drop, Parameter Fitting PBR):  
  Packed-bed catalyst weight (W)  
    Uses curve_fit first to get k, K_B, K_T;  
    pressure drop (alpha or Ergun parameters)  

  Step 1 (Pre-Solver): Use scipy.optimize.curve_fit to fit kinetic constants ($k, K_B, K_T$) from experimental rates  
    Watch out for scaling factors (e.g., 10^(-10)) and unit conversions g_cat to kg_cat, s to min  
    State Vector z: [FT, FH2, FB, FM, FI, p]

  Uses partial pressures P_i = (F_i/(F_tot)*p*P_0) instead of concentration/volumetric flow;  
    adds dimensionless pressure p = P/P_0 to state vector z with dp/dW = -alpha/(2p).  
    See alpha = ? Put p in z (starting at 1.0) and pass P0 in args.  

  Don't see alpha, and instead see viscosity, particle size, and porosity?  
    Put P directly in z (starting at P0) and write the full Ergun equation for dP/dW.# Tutorial 2, Problem 1: Liquid PFR — Adiabatic, Reversible (dV)  

  Liquid phase, no pressure drop, reversible, adiabatic ($Q = 0$)  
   Dependent Variable Vector z: [Fi,....., T]  
    Volumetric flow: Constant (v = v0 = F_A0/C_A0), passed via args=(v0,)  
     Concentrations: C_i = F_i / v0  
      Kinetics: Both k(T) (Arrhenius) and K_C(T) (Van 't Ho) vary with T  
       Energy Balance: dTdV = (r_A \Delta H)/(\sum F_i C_pi)  
